# DistilBERT PII Detection

## Goal

Fine-tune DistilBERT for binary classification of privacy-sensitive prompts.

Labels:
- 0 = safe (no PII)
- 1 = privacy sensitive (contains PII)

Evaluation metrics match the classical ML baseline: precision, recall, F1, and confusion matrix per class.

In [1]:
# Cell 2 -- Imports
import pandas as pd
import numpy as np
import torch
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset

In [2]:
# Cell 3 -- confirm GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

Using device: cuda
GPU: NVIDIA GeForce RTX 4090


In [3]:
# Cell 4 -- load cleaned frozen splits generated by data_split.py
train = pd.read_parquet('../data_splits/train.parquet')
val = pd.read_parquet('../data_splits/val.parquet')
test = pd.read_parquet('../data_splits/test.parquet')

# sanity check
print("train:", train.shape)
print("val:", val.shape)
print("test:", test.shape)
print("\nLabel balance (train):")
print(train['label'].value_counts(normalize=True) * 100)

train: (260338, 3)
val: (32542, 3)
test: (32543, 3)

Label balance (train):
label
1    67.377025
0    32.622975
Name: proportion, dtype: float64


## Tokenization

DistilBERT requires text to be tokenized using its own tokenizer. We use `DistilBertTokenizerFast` with truncation and padding to a max length of 128 tokens, which covers the majority of examples in this dataset given the average prompt length from EDA (~157 chars / ~20 words for sensitive examples).

In [4]:
# Cell 6 -- load DistilBERT tokenizer
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

# tokenize all three splits
def tokenize(texts, labels, max_length=128):
    encodings = tokenizer(
        list(texts),
        truncation=True,
        padding=True,
        max_length=max_length
    )
    return encodings, list(labels)

train_encodings, train_labels = tokenize(train['text'], train['label'])
val_encodings, val_labels = tokenize(val['text'], val['label'])
test_encodings, test_labels = tokenize(test['text'], test['label'])

print("Tokenization complete.")

Tokenization complete.


In [5]:
# Cell 7 -- custom dataset class for DistilBERT
class PIIDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = PIIDataset(train_encodings, train_labels)
val_dataset = PIIDataset(val_encodings, val_labels)
test_dataset = PIIDataset(test_encodings, test_labels)

print(f"Train: {len(train_dataset)} examples")
print(f"Val: {len(val_dataset)} examples")
print(f"Test: {len(test_dataset)} examples")

Train: 260338 examples
Val: 32542 examples
Test: 32543 examples


## Model

We load DistilBERT with a binary classification head (`num_labels=2`). The model is fine-tuned on the training split and evaluated against the val split during training.

In [6]:
# Cell 9 -- load DistilBERT with binary classification head
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2
)

model.to(device)
print("Model loaded and moved to:", device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded and moved to: cuda


In [7]:
# Cell 10 -- define metrics function for Trainer
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    report = classification_report(labels, preds, target_names=['Safe', 'PII'], output_dict=True)
    return {
        'f1_pii': report['PII']['f1-score'],
        'recall_pii': report['PII']['recall'],
        'f1_safe': report['Safe']['f1-score'],
        'f1_macro': report['macro avg']['f1-score'],
        'accuracy': report['accuracy']
    }

In [8]:
# Cell 11 -- configure training arguments
training_args = TrainingArguments(
    output_dir='../results/distilbert',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='../results/distilbert/logs',
    logging_steps=100,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_pii',
    fp16=torch.cuda.is_available(),
    seed=42
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [9]:
# Cell 12 -- initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

print("Trainer initialized.")

Trainer initialized.


## Training

Fine-tune DistilBERT on the training split. Evaluation runs after each epoch against the validation split. Best model checkpoint is loaded at the end based on PII F1 score, since false negatives (sensitive prompts missed) are the primary concern for this project.

## Limitations

- **English-only model**: `distilbert-base-uncased` is trained on English text only. The AI4Privacy dataset contains multilingual examples (German, Spanish, French, Italian, Dutch). Performance on non-English examples may be lower than overall metrics suggest. A future improvement would be to switch to `distilbert-base-multilingual-cased` and retrain.
- **HTML tokens**: The tokenizer treats HTML tags as regular tokens. HTML-heavy examples may introduce noise.

In [10]:
# Cell 14 -- fine-tune DistilBERT
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Pii,F1 Safe,F1 Macro,Accuracy
1,0.137306,0.127057,0.965563,0.928022,0.946792,0.953414
2,0.084335,0.113547,0.970868,0.940760,0.955814,0.960943
3,0.048021,0.151595,0.972654,0.943547,0.958100,0.963155


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=24408, training_loss=0.1134018382232762, metrics={'train_runtime': 515.2115, 'train_samples_per_second': 1515.91, 'train_steps_per_second': 47.375, 'total_flos': 2.5864723223479296e+16, 'train_loss': 0.1134018382232762, 'epoch': 3.0})

## Evaluation

Evaluate the best checkpoint against the validation set. Metrics match the classical ML baseline: precision, recall, F1 per class, and confusion matrix. Special attention on PII recall -- false negatives represent sensitive prompts incorrectly classified as safe.

In [11]:
# Cell 16 -- evaluate best model on validation set
val_predictions = trainer.predict(val_dataset)
val_preds = val_predictions.predictions.argmax(-1)

print(classification_report(val_labels, val_preds, target_names=['Safe', 'PII']))
print(confusion_matrix(val_labels, val_preds))

              precision    recall  f1-score   support

        Safe       0.94      0.94      0.94     10616
         PII       0.97      0.97      0.97     21926

    accuracy                           0.96     32542
   macro avg       0.96      0.96      0.96     32542
weighted avg       0.96      0.96      0.96     32542

[[10020   596]
 [  603 21323]]


## Test Set Evaluation

Run on the held-out test set only after validation results are satisfactory. Do not use test results to tune the model.

In [12]:
# Cell 18 -- evaluate best model on held-out test set
test_predictions = trainer.predict(test_dataset)
test_preds = test_predictions.predictions.argmax(-1)

print(classification_report(test_labels, test_preds, target_names=['Safe', 'PII']))
print(confusion_matrix(test_labels, test_preds))

              precision    recall  f1-score   support

        Safe       0.94      0.94      0.94     10617
         PII       0.97      0.97      0.97     21926

    accuracy                           0.96     32543
   macro avg       0.96      0.96      0.96     32543
weighted avg       0.96      0.96      0.96     32543

[[10031   586]
 [  624 21302]]
